# HUF Triad — Vol 0, Notebook 3
# 🍞 Sourdough Baker: Real Data, Real Detection

**Audience**: Anyone who finished Notebooks 1–2

**What you will learn**:
- HUF applied to real sourdough fermentation data
- Why yeast has **ultra-high leverage** (833×)
- How HUF detects when your bread recipe silently changes
- The **Quality Factor (Q)** — why some ingredients matter more per gram

**Time**: ~20 minutes

**Data source**: Three-domain confirmation System A (Sourdough, p=0.021)

---

> *Previous*: `01_my_backpack.ipynb`  
> *Next*: `03_city_explorer.ipynb` (Toronto TTC transit data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
print('Ready! Let\'s bake some bread \u2014 with mathematics.')

## 1. A Sourdough Loaf as a Portfolio

A sourdough loaf has a **budget ceiling** of 1,000g (total mass).  
Five ingredients share that ceiling. Each ingredient's mass is a **share** of the whole.

This is real data from the HUF three-domain confirmation (System A).

In [ ]:
# Real sourdough portfolio data (System A)
ingredients = ['Flour', 'Water', 'Starter', 'Salt', 'Yeast']
mass_g = [500, 350, 130, 18, 1.2]  # grams
total_mass = sum(mass_g)  # ~999.2g

# Normalize to shares
shares = [m / total_mass for m in mass_g]
leverage = [1 / s for s in shares]

print(f'Budget ceiling: {total_mass:.1f}g\n')
print(f'{"Ingredient":<12} {"Mass (g)":<10} {"Share (ρ)":<12} {"Leverage":<10} {"Q class":<12}')
print('-' * 60)

q_classes = ['Low Q', 'Low Q', 'Medium Q', 'Medium Q', 'Ultra-high Q']
for ing, m, s, l, q in zip(ingredients, mass_g, shares, leverage, q_classes):
    flag = ' \u26a0\ufe0f' if l > 100 else ''
    print(f'{ing:<12} {m:<10.1f} {s:<12.4f} {l:<10.1f} {q:<12}{flag}')

print(f'\n\u2211\u03c1 = {sum(shares):.4f}  \u2714 Unity constraint holds.')
print(f'\nYeast: only {mass_g[4]}g out of {total_mass:.0f}g, but leverage = {leverage[4]:.0f}')
print('  \u2192 Remove the yeast and the bread won\'t rise. That\'s what leverage MEANS.')

In [ ]:
# Visualize: the logarithmic reality of leverage
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#D4A574', '#45B7D1', '#8B6914', '#95A5A6', '#E74C3C']

# Shares (linear) — yeast is invisible
ax1.bar(ingredients, [s*100 for s in shares], color=colors, edgecolor='white')
ax1.set_ylabel('Share (%)', fontsize=13)
ax1.set_title('Shares: Yeast is Almost Invisible', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 55)

# Leverage (log scale) — yeast dominates
ax2.bar(ingredients, leverage, color=colors, edgecolor='white')
ax2.set_ylabel('Leverage (1/ρ)', fontsize=13)
ax2.set_title('Leverage: Yeast Dominates', fontsize=14, fontweight='bold')
ax2.set_yscale('log')
ax2.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='Ultra-high Q threshold')
ax2.legend()

plt.tight_layout()
plt.show()
print('Left: by MASS, yeast barely exists. Right: by LEVERAGE, yeast is king.')
print('This is FM-1 (Ratio Blindness): looking at mass alone misses what matters.')

## 2. Detecting Silent Drift in Your Bread

Suppose you declare your recipe (the **declared weights**).  
Then over several bakes, things silently shift: you use a bit less starter,  
a bit more water. Nobody decided this — it just happened.

Let's simulate 6 baking cycles and watch HUF detect the drift.

In [ ]:
# Simulate 6 baking cycles with gradual silent drift
np.random.seed(42)

declared = np.array([0.5004, 0.3503, 0.1301, 0.0180, 0.0012])  # declared shares

# Each cycle, small random perturbations simulate measurement error + drift
cycles = 6
drift_rate = np.array([0.000, 0.005, -0.004, -0.001, 0.000])  # water up, starter down

all_observed = []
all_mdg = []

print(f'{"Cycle":<8} {"MDG (pp)":<10} {"Status":<20} {"Biggest Drift"}')
print('-' * 60)

for cycle in range(cycles):
    # Apply cumulative drift + small noise
    observed = declared + drift_rate * (cycle + 1) + np.random.normal(0, 0.001, 5)
    observed = np.abs(observed)
    observed = observed / observed.sum()  # re-normalize to simplex
    
    all_observed.append(observed.copy())
    
    # Compute MDG
    gaps = np.abs(declared - observed)
    mdg = np.mean(gaps)
    all_mdg.append(mdg)
    
    # Status
    if mdg * 100 < 2:
        status = '\u2714 Ground state'
    elif mdg * 100 < 5:
        status = '\u26a0\ufe0f Action window'
    else:
        status = '\ud83d\udea8 Active drift!'
    
    biggest = ingredients[np.argmax(gaps)]
    print(f'{cycle+1:<8} {mdg*100:<10.2f} {status:<20} {biggest} ({gaps[np.argmax(gaps)]*100:.2f}pp)')

print(f'\nDrift accumulated over {cycles} cycles. Water share crept up, starter crept down.')

In [ ]:
# Visualize MDG trajectory
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MDG over time
cycle_nums = range(1, cycles + 1)
ax1.plot(cycle_nums, [m*100 for m in all_mdg], 'o-', color='#E74C3C', linewidth=2, markersize=8)
ax1.axhline(y=2, color='green', linestyle='--', alpha=0.5, label='Ground state (< 2pp)')
ax1.axhline(y=5, color='orange', linestyle='--', alpha=0.5, label='Active drift (> 5pp)')
ax1.fill_between(cycle_nums, 0, 2, alpha=0.1, color='green')
ax1.fill_between(cycle_nums, 2, 5, alpha=0.1, color='orange')
ax1.fill_between(cycle_nums, 5, max([m*100 for m in all_mdg]) + 1, alpha=0.1, color='red')
ax1.set_xlabel('Bake Cycle', fontsize=13)
ax1.set_ylabel('Mean Drift Gap (pp)', fontsize=13)
ax1.set_title('MDG Trajectory: Drift Accumulating', fontsize=14, fontweight='bold')
ax1.legend()

# Ingredient trajectories
obs_array = np.array(all_observed)
for i, (ing, col) in enumerate(zip(ingredients, colors)):
    ax2.plot(cycle_nums, obs_array[:, i], 'o-', color=col, label=ing, linewidth=2)
    ax2.axhline(y=declared[i], color=col, linestyle=':', alpha=0.3)

ax2.set_xlabel('Bake Cycle', fontsize=13)
ax2.set_ylabel('Observed Share (ρ)', fontsize=13)
ax2.set_title('Ingredient Shares Over Time', fontsize=14, fontweight='bold')
ax2.legend(loc='center right', fontsize=9)

plt.tight_layout()
plt.show()
print('Left: MDG climbs as drift accumulates. Right: water creeps up, starter creeps down.')
print('Dotted lines = declared shares. Solid lines = observed. The gap = silent drift.')

## 3. The Quality Factor: Why Yeast Matters More Per Gram

The **Quality Factor (Q)** measures how sensitive an element is relative to its observation window.

- **Low Q** (flour, water): broadly functional, visible in any observation. Change 10g and you barely notice.
- **Ultra-high Q** (yeast): contributes in narrow bursts (fermentation window). Change 0.5g and the bread fails.

Snapshot Error (FM-3) happens when you measure a high-Q element at the wrong time  
and conclude it's unimportant.

In [ ]:
# Q-factor illustration
q_data = {
    'Flour':   {'T': 'Continuous',    'B': 'Always visible', 'Q': 1.6, 'class': 'Low Q'},
    'Water':   {'T': 'Continuous',    'B': 'Always visible', 'Q': 2.9, 'class': 'Low Q'},
    'Starter': {'T': '4-12 hours',    'B': '~1 hour active', 'Q': 7.7, 'class': 'Medium Q'},
    'Salt':    {'T': 'Continuous',     'B': 'Always present', 'Q': 55.6, 'class': 'Medium Q'},
    'Yeast':   {'T': '4-12 hours',    'B': '0.01-0.02 hours', 'Q': 833.0, 'class': 'Ultra-high Q'},
}

print(f'{"Ingredient":<12} {"Characteristic Period":<22} {"Observation Window":<20} {"Q":<8} {"Class"}')
print('-' * 75)
for ing, d in q_data.items():
    print(f'{ing:<12} {d["T"]:<22} {d["B"]:<20} {d["Q"]:<8.1f} {d["class"]}')

print(f'\nYeast Q = 833: its characteristic period is ~500\u00d7 its observation window.')
print('If you measure at the wrong time, you\'ll think yeast does nothing.')
print('That\'s Snapshot Error (FM-3) \u2014 the phase read as contribution.')

## 4. Try It Yourself: Design a Recipe

Create your own bread recipe and see what HUF says about it.

In [ ]:
# ✏️ DESIGN YOUR RECIPE (grams)
my_recipe = {
    'Flour': 600,
    'Water': 400,
    'Starter': 80,
    'Salt': 12,
    'Yeast': 2.0,  # Try: 0.5 (less yeast) or 5.0 (more yeast)
}

# ------- computation -------
my_total = sum(my_recipe.values())
my_shares = {k: v/my_total for k, v in my_recipe.items()}
my_leverage = {k: 1/v for k, v in my_shares.items()}

print(f'Total mass: {my_total:.1f}g\n')
for ing in my_recipe:
    s = my_shares[ing]
    l = my_leverage[ing]
    flag = ' \u26a0\ufe0f ULTRA-HIGH Q' if l > 100 else (' \u26a0\ufe0f HIGH Q' if l > 10 else '')
    print(f'{ing:<12} {my_recipe[ing]:>7.1f}g  \u03c1={s:.4f}  leverage={l:.1f}{flag}')

print(f'\n\u2211\u03c1 = {sum(my_shares.values()):.4f}')

# Compare to reference recipe
ref = {'Flour': 0.5004, 'Water': 0.3503, 'Starter': 0.1301, 'Salt': 0.0180, 'Yeast': 0.0012}
gaps = [abs(my_shares[k] - ref[k]) for k in my_shares]
mdg = np.mean(gaps)
print(f'MDG vs reference recipe: {mdg*100:.2f}pp')

## 5. Summary: From Bread to Universal Monitoring

| Sourdough Concept | HUF Term | Real-World Parallel |
|-------------------|----------|--------------------|
| 1,000g total mass | Budget Ceiling | 93,000 ha total wetland area |
| Flour at 50% | Low-Q, low-leverage element | Dominant conservation site |
| Yeast at 0.12% | Ultra-high Q, leverage 833 | Rare endemic species site |
| Recipe silently changes | Silent Drift (FM-2) | Budget allocation creeps |
| Measure yeast at wrong time | Snapshot Error (FM-3) | Survey outside breeding season |
| One ingredient dominates | Concentration Trap (FM-4) | Single site gets all funding |

**The sourdough loaf was the first system where HUF was confirmed** (p=0.021, Pettitt test).  
The same mathematics works for wetlands, satellites, transit systems, and hard drive fleets.

---

### Next Steps

- **Notebook 4**: Toronto TTC data → `03_city_explorer.ipynb`
- **Notebook 5**: Planck satellite data → `04_star_listener.ipynb`  
- **Vol 2**: All three domain confirmations → *Case Studies*
- **Vol 4**: Ecological applications → *Ramsar & Environmental*